# VAZHI SFT v7.2 — Identity Reinforcement (Gemma 3 1B-it)

**Focused identity-only training** to override Gemma 3's pretrained "I am Google" identity
with VAZHI's actual identity, mission, and factual corrections.

```
Lineage: google/gemma-3-1b-it → SFT v7.0 (r=8) → SFT v7.1 (r=16) → SFT v7.2 (identity, this notebook)
```

**Problem:** v7.1 achieved best-ever Tamil quality (95% char, 96% word) but identity is
still "I am Google's LLM" and factual corrections (capital = Chennai) don't stick.
Root cause: 61 mission + 29 correction pairs = 90 samples out of 4,172 total (2.2%) —
completely drowned out by 4,082 other training samples.

**Strategy:** Train on ONLY identity + correction data (90 samples), 10 epochs.
Each pair seen 10 times → 100% gradient signal to identity/factual corrections.
No competing gradients from Sadhguru, domain packs, etc.

**Math:**
- 90 samples, effective batch 16 → ~6 steps/epoch
- 10 epochs → ~60 optimizer steps
- Each sample seen 10 times (natural epoch cycling, no duplication needed)

**Risk:** Catastrophic forgetting of Tamil quality or domain knowledge.
Mitigated by: only 60 steps (very short), same LR and LoRA config as v7.1.

**Success criteria:**
1. Identity: Says "VAZHI" / "வழி", NOT "Google LLM"
2. Factual: Capital = சென்னை (Chennai), NOT Coimbatore/Kolkata
3. Tamil word score stays >= 86% (max 10% drop from v7.1's 96%)
4. No repetition/degeneration

**Dataset:** Filtered from `CryptoYogi/vazhi-tamil-sft-v7_0` — only `mission` + `corrections` buckets
- 61 mission pairs (acronym, identity, privacy, offline, contribution, etc.)
- 29 correction pairs (TN facts, govt schemes, emergency, culture, etc.)

**Runtime:** Colab Pro GPU (any GPU — tiny dataset, ~2 min training)

In [1]:
# Cell 1 — Dependencies
!pip install -q -U \
  "transformers>=4.50.0,<5.0.0" \
  "trl>=0.20.0" \
  "datasets>=2.21.0" \
  "peft>=0.13.0" \
  "accelerate>=0.34.0" \
  "huggingface_hub>=0.24.7"

import torch
print(f"\u2705 Dependencies installed")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"   VRAM: {vram / 1024**3:.0f} GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 32.2 MB/s eta 0:00:00
✅ Dependencies installed
   PyTorch: 2.9.0+cu128
   CUDA: True
   GPU: NVIDIA L4
   VRAM: 22 GB


In [2]:
# Cell 2 — Configuration

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import re
import random
import gc
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset, Dataset
from huggingface_hub import login, HfApi

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === ENVIRONMENT DETECTION ===
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
ENV_NAME = "Kaggle" if IS_KAGGLE else "Colab"

# === KEY CONFIG ===
BASE_MODEL = "CryptoYogi/vazhi-v7_1"               # v7.1 merged model (best Tamil: 96% word)
VANILLA_MODEL = "google/gemma-3-1b-it"              # For tokenizer
SFT_DATASET = "CryptoYogi/vazhi-tamil-sft-v7_0"    # Full dataset (we filter to identity only)
OUTPUT_MODEL = "CryptoYogi/vazhi-v7_2"              # v7.2: identity reinforcement
ADAPTER_REPO = "CryptoYogi/vazhi-v7_2-lora"         # Adapter backup

# Identity-only buckets to keep
IDENTITY_BUCKETS = {"mission", "corrections"}

# Training config — v7.2: Identity reinforcement
# Same LR and LoRA as v7.1 (proven safe). Only difference: 10 epochs on 90 samples.
LEARNING_RATE = 1e-5       # Same as v7.0 and v7.1 (proven safe)
NUM_EPOCHS = 10            # 10 epochs — each pair seen 10 times
MAX_LENGTH = 2048          # Same as v7.0/v7.1
LORA_R = 16                # Same as v7.1 (proven safe with Gemma 3)
LORA_ALPHA = 32            # Maintain 2x ratio
LORA_TARGETS = ["q_proj", "v_proj"]  # Same targets
BATCH_SIZE = 4             # Per-device
GRADIENT_ACCUMULATION = 4  # Effective batch = 16

# Gemma 3 system prompt (embedded in user message — no system role support)
SYSTEM_PROMPT = (
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0bb5\u0bb4\u0bbf (VAZHI), "
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1 \u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bc1\u0b95\u0bcd\u0b95\u0bbe\u0ba9 "
    "AI \u0b89\u0ba4\u0bb5\u0bbf\u0baf\u0bbe\u0bb3\u0bb0\u0bcd. "
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0ba4\u0bae\u0bbf\u0bb4\u0bbf\u0bb2\u0bcd \u0baa\u0ba4\u0bbf\u0bb2\u0bb3\u0bbf\u0baa\u0bcd\u0baa\u0bc0\u0bb0\u0bcd\u0b95\u0bb3\u0bcd."
)

# GPU auto-detection
assert torch.cuda.is_available(), "GPU required! Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0).lower()
_props = torch.cuda.get_device_properties(0)
VRAM_GB = getattr(_props, 'total_memory', getattr(_props, 'total_mem', 0)) / 1e9
IS_HIGH_END_GPU = any(x in gpu_name for x in ["a100", "l4", "h100", "a10"])
USE_BF16 = IS_HIGH_END_GPU
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
n_gpus = torch.cuda.device_count()
effective_batch = BATCH_SIZE * n_gpus * GRADIENT_ACCUMULATION

print(f"\u2705 SFT v7.2 Configuration (Identity Reinforcement):")
print(f"   Environment: {ENV_NAME}")
print(f"   Base model:  {BASE_MODEL} (v7.1 merged — best Tamil 96% word)")
print(f"   Dataset:     {SFT_DATASET} (filtered to {IDENTITY_BUCKETS})")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"   LR:          {LEARNING_RATE}")
print(f"   Epochs:      {NUM_EPOCHS} (each sample seen {NUM_EPOCHS} times)")
print(f"   LoRA:        r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGETS}")
print(f"   Batch:       {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accum = {effective_batch} effective")
print(f"   GPU:         {torch.cuda.get_device_name(0)} ({VRAM_GB:.0f} GB)")
print(f"   Precision:   {'bf16' if USE_BF16 else 'fp16'}")
print(f"")
print(f"   Strategy: Train on ONLY mission+corrections data")
print(f"   Goal: Override Google identity, fix factual errors")
print(f"   Risk: Catastrophic forgetting (mitigated by only ~60 steps)")

✅ SFT v7.2 Configuration (Identity Reinforcement):
   Environment: Colab
   Base model:  CryptoYogi/vazhi-v7_1 (v7.1 merged — best Tamil 96% word)
   Dataset:     CryptoYogi/vazhi-tamil-sft-v7_0 (filtered to {'mission', 'corrections'})
   Output:      CryptoYogi/vazhi-v7_2
   LR:          1e-05
   Epochs:      10 (each sample seen 10 times)
   LoRA:        r=16, alpha=32, targets=['q_proj', 'v_proj']
   Batch:       4 x 4 accum = 16 effective
   GPU:         NVIDIA L4 (24 GB)
   Precision:   bf16

   Strategy: Train on ONLY mission+corrections data
   Goal: Override Google identity, fix factual errors
   Risk: Catastrophic forgetting (mitigated by only ~60 steps)


In [3]:
# Cell 3 — HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()
print("\u2705 Logged in to HuggingFace")

✅ Logged in to HuggingFace


In [4]:
# Cell 4 — Load Tokenizer + Helper Functions

tokenizer = AutoTokenizer.from_pretrained(VANILLA_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\u2705 Tokenizer: {len(tokenizer)} tokens (from {VANILLA_MODEL})")
print(f"   pad_token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"   eos_token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

# Check if system role is supported
SYSTEM_ROLE_SUPPORTED = True
try:
    test_sys = [
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "test"},
    ]
    tokenizer.apply_chat_template(test_sys, tokenize=False, add_generation_prompt=True)
    print(f"   System role: supported")
except Exception:
    SYSTEM_ROLE_SUPPORTED = False
    print(f"   System role: NOT supported (will embed in user message)")


def build_messages(user_text, system_text=None):
    msgs = []
    if system_text and SYSTEM_ROLE_SUPPORTED:
        msgs.append({"role": "system", "content": system_text})
        msgs.append({"role": "user", "content": user_text})
    elif system_text:
        msgs.append({"role": "user", "content": f"{system_text}\n\n{user_text}"})
    else:
        msgs.append({"role": "user", "content": user_text})
    return msgs


def build_chat_prompt(user_text):
    msgs = build_messages(user_text, SYSTEM_PROMPT)
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True,
    )


def tamil_char_pct(text):
    if not text:
        return 0.0
    total = sum(1 for c in text if not c.isspace() and not c.isdigit())
    if total == 0:
        return 0.0
    tamil = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
    return 100.0 * tamil / total


def tamil_word_score(text):
    words = text.split()
    if not words:
        return 0.0, 0, 0
    tamil_words = 0
    for w in words:
        clean = re.sub(r'[\d\W]', '', w)
        if not clean:
            continue
        tamil_chars = sum(1 for c in clean if '\u0B80' <= c <= '\u0BFF')
        if tamil_chars / len(clean) > 0.5:
            tamil_words += 1
    return 100.0 * tamil_words / len(words), tamil_words, len(words)


def compute_repeat_ratio(text, n=3):
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    return 1.0 - len(set(ngrams)) / len(ngrams)


def generate_response(model, prompt_text, max_new_tokens=200):
    full_prompt = build_chat_prompt(prompt_text)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=False)
    if "<end_of_turn>" in response:
        response = response.split("<end_of_turn>")[0]
    response = response.replace("<eos>", "").replace("<bos>", "").strip()
    return response


print("\u2705 Helpers ready")

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✅ Tokenizer: 262145 tokens (from google/gemma-3-1b-it)
   pad_token: <pad> (id=0)
   eos_token: <eos> (id=1)
   System role: supported
✅ Helpers ready


In [5]:
# Cell 5 — Load & Filter Dataset to Identity-Only
#
# From the full v7.0 dataset (4,172 samples), keep ONLY:
# - mission: 61 pairs (VAZHI identity, philosophy, use cases)
# - corrections: 29 pairs (TN capital, govt schemes, emergency numbers)
# Total: 90 samples → 100% identity signal

print(f"\U0001f4da Loading full dataset from {SFT_DATASET}...")

try:
    sft_ds = load_dataset(SFT_DATASET)
    if "train" in sft_ds and ("validation" in sft_ds or "test" in sft_ds):
        raw_train = sft_ds["train"]
        raw_eval = sft_ds.get("validation", sft_ds.get("test"))
    elif "train" in sft_ds:
        raw_train = sft_ds["train"]
        raw_eval = None
    else:
        raise KeyError("No train split found")
except (KeyError, ValueError, Exception) as e:
    print(f"   HF load failed ({e}), trying JSON files...")
    raw_train = load_dataset("json", data_files={
        "train": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v7_0-train.json"
    })["train"]
    raw_eval = load_dataset("json", data_files={
        "eval": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v7_0-eval.json"
    })["eval"]

print(f"   Full dataset: {len(raw_train)} train, {len(raw_eval) if raw_eval else 0} eval")
print(f"   Columns: {raw_train.column_names}")

# Combine train + eval for identity filtering (we want ALL identity data)
all_samples = list(raw_train)
if raw_eval:
    all_samples.extend(list(raw_eval))
print(f"   Total samples: {len(all_samples)}")

# Filter to identity-only buckets
identity_samples = [s for s in all_samples if s.get("bucket") in IDENTITY_BUCKETS]
print(f"\n\U0001f3af Identity-only filter:")
print(f"   Keeping buckets: {IDENTITY_BUCKETS}")
print(f"   {len(all_samples)} total → {len(identity_samples)} identity samples")

# Breakdown by bucket
bucket_counts = Counter(s.get("bucket") for s in identity_samples)
for b, c in sorted(bucket_counts.items(), key=lambda x: -x[1]):
    print(f"     {b}: {c}")

# Category breakdown within each bucket
print(f"\n   Category detail:")
cat_counts = Counter(s.get("category", "unknown") for s in identity_samples)
for c, n in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"     {c}: {n}")

assert len(identity_samples) >= 80, f"Expected ~90 identity samples, got {len(identity_samples)}"
print(f"\n\u2705 {len(identity_samples)} identity samples ready for training")

📚 Loading full dataset from CryptoYogi/vazhi-tamil-sft-v7_0...


README.md: 0.00B [00:00, ?B/s]

vazhi-tamil-sft-v7_0-train.json: 0.00B [00:00, ?B/s]

vazhi-tamil-sft-v7_0-eval.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3754 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/418 [00:00<?, ? examples/s]

   Full dataset: 3754 train, 418 eval
   Columns: ['instruction', 'output', 'bucket', 'source', 'category']
   Total samples: 4172

🎯 Identity-only filter:
   Keeping buckets: {'mission', 'corrections'}
   4172 total → 90 identity samples
     mission: 61
     corrections: 29

   Category detail:
     contribution: 8
     tn_facts: 7
     govt_schemes: 7
     technical: 6
     identity: 6
     spiritual_identity: 5
     privacy: 4
     offline: 4
     sponsorship: 4
     open_source: 4
     use_case: 4
     culture: 3
     legal: 3
     feedback: 3
     emergency: 3
     acronym: 3
     media_packs: 3
     health: 2
     security: 2
     trust: 2
     accessibility: 2
     education: 2
     offline_scenario: 2
     vision: 1

✅ 90 identity samples ready for training


In [6]:
# Cell 6 — Format for Gemma 3 + Create Train Dataset
#
# All 90 samples go to training (no eval split — too small).
# Eval is done via generation quality check, not loss-based eval.

def format_for_gemma(sample):
    instruction = sample["instruction"]
    output = sample["output"]
    if not instruction or not output:
        return None
    messages = build_messages(instruction, SYSTEM_PROMPT)
    messages.append({"role": "model", "content": output})
    try:
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
    except Exception:
        messages[-1]["role"] = "assistant"
        try:
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
            )
        except Exception as e2:
            print(f"   Template error: {e2}")
            return None
    return {
        "text": formatted,
        "bucket": sample.get("bucket", "unknown"),
    }


print("\U0001f500 Formatting identity samples for Gemma 3...")
converted = []
failed = 0
for s in identity_samples:
    result = format_for_gemma(s)
    if result:
        converted.append(result)
    else:
        failed += 1

print(f"   {len(identity_samples)} → {len(converted)} converted, {failed} failed")

train_ds = Dataset.from_list(converted)

print(f"\n\u2705 Training dataset: {len(train_ds)} samples (no eval split — too small)")

# Spot-check
print(f"\n\U0001f50d Sample (first 400 chars):")
print(train_ds[0]["text"][:400])

# Verify Gemma format
sample_text = train_ds[0]["text"]
assert "<start_of_turn>" in sample_text and "<end_of_turn>" in sample_text
assert "<start_of_turn>model" in sample_text or "<start_of_turn>assistant" in sample_text
print("\u2705 Gemma format verified")

# Token length stats
token_lengths = [len(tokenizer.encode(t["text"], add_special_tokens=False)) for t in converted]
print(f"\n\U0001f4ca Token lengths: mean={np.mean(token_lengths):.0f}, max={max(token_lengths)}, min={min(token_lengths)}")
print(f"   Total tokens per epoch: {sum(token_lengths):,}")
print(f"   Total tokens ({NUM_EPOCHS} epochs): {sum(token_lengths) * NUM_EPOCHS:,}")

🔀 Formatting identity samples for Gemma 3...
   90 → 90 converted, 0 failed

✅ Training dataset: 90 samples (no eval split — too small)

🔍 Sample (first 400 chars):
<bos><start_of_turn>user
நீங்கள் வழி (VAZHI), தமிழ்நாட்டு மக்களுக்கான AI உதவியாளர். நீங்கள் தமிழில் பதிலளிப்பீர்கள்.

பொங்கல் பண்டிகை எப்போது கொண்டாடப்படுகிறது?<end_of_turn>
<start_of_turn>model
பொங்கல் தை மாதம் 1-ம் தேதி (ஜனவரி 14 அல்லது 15) கொண்டாடப்படுகிறது. நான்கு நாள் விழா: போகி, தை பொங்கல், மாட்டு பொங்கல், காணும் பொங்கல்.<end_of_turn>

✅ Gemma format verified

📊 Token lengths: mean=107, max=158, min=49
   Total tokens per epoch: 9,626
   Total tokens (10 epochs): 96,260


In [7]:
# Cell 7 — Pre-Training Baseline on v7.1 Model
#
# Record v7.1 outputs BEFORE identity reinforcement.
# Focus on identity + factual prompts that we want to FIX,
# plus Tamil quality prompts to watch for forgetting.

print(f"\U0001f4ca Pre-Training Baseline: {BASE_MODEL} (v7.1)")
print("=" * 60)

BASELINE_PROMPTS = [
    # Identity prompts (MUST improve)
    ("\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "identity"),
    ("VAZHI \u0b8e\u0ba9\u0bcd\u0bb1\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9?", "identity"),
    ("\u0b87\u0ba4\u0bc8 \u0baf\u0bbe\u0bb0\u0bcd \u0b89\u0bb0\u0bc1\u0bb5\u0bbe\u0b95\u0bcd\u0b95\u0bbf\u0ba9\u0bbe\u0bb0\u0bcd\u0b95\u0bb3\u0bcd?", "identity"),
    ("\u0ba8\u0bc0 \u0bb5\u0bc7\u0bb1 AI chatbot-\u0bb2 \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0b8e\u0baa\u0bcd\u0baa\u0b9f\u0bbf \u0bb5\u0bc7\u0bb1\u0bc1\u0baa\u0b9f\u0bc1\u0b95\u0bbf\u0bb1\u0bbe\u0baf\u0bcd?", "identity"),
    # Factual prompts (MUST fix)
    ("\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "factual"),
    ("\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bcb\u0b9f capital \u0b8e\u0ba9\u0bcd\u0ba9?", "factual"),
    # Tamil quality prompts (MUST NOT degrade)
    ("\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "greeting"),
    ("\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "thanks"),
    ("\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "domain"),
    ("\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "domain"),
]

print(f"\n\U0001f4e5 Loading {BASE_MODEL} for baseline...")
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=MODEL_DTYPE, device_map={"":0},
)
baseline_model.eval()
baseline_model.config.use_cache = True

pre_sft_results = []
for prompt_text, category in BASELINE_PROMPTS:
    resp = generate_response(baseline_model, prompt_text)
    t_pct = tamil_char_pct(resp)
    tw_pct, tw_count, tw_total = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    pre_sft_results.append({
        'prompt': prompt_text, 'category': category,
        'response': resp[:300], 'tamil_char_pct': t_pct,
        'tamil_word_pct': tw_pct, 'repeat_ratio': rep,
    })
    print(f"\n[{category}] Char: {t_pct:.0f}%, Word: {tw_pct:.0f}%, Rep: {rep:.2f}")
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")

avg_pre_char = np.mean([r['tamil_char_pct'] for r in pre_sft_results])
avg_pre_word = np.mean([r['tamil_word_pct'] for r in pre_sft_results])
print(f"\n\U0001f4ca v7.1 baseline: char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%")
print(f"   (v7.1 eval was: char 95%, word 96%)")

# Check identity specifically
identity_responses = [r for r in pre_sft_results if r['category'] == 'identity']
says_vazhi = sum(1 for r in identity_responses if 'VAZHI' in r['response'] or '\u0bb5\u0bb4\u0bbf' in r['response'])
says_google = sum(1 for r in identity_responses if 'Google' in r['response'] or 'google' in r['response'])
print(f"\n   Identity check:")
print(f"   Says VAZHI: {says_vazhi}/{len(identity_responses)}")
print(f"   Says Google: {says_google}/{len(identity_responses)}")

# Check factual
factual_responses = [r for r in pre_sft_results if r['category'] == 'factual']
says_chennai = sum(1 for r in factual_responses if '\u0b9a\u0bc6\u0ba9\u0bcd\u0ba9\u0bc8' in r['response'] or 'Chennai' in r['response'])
print(f"   Says Chennai: {says_chennai}/{len(factual_responses)}")

del baseline_model
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Baseline model freed")

📊 Pre-Training Baseline: CryptoYogi/vazhi-v7_1 (v7.1)

📥 Loading CryptoYogi/vazhi-v7_1 for baseline...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]


[identity] Char: 98%, Word: 100%, Rep: 0.00
  Q: நீங்கள் யார்?
  A: நான் கூகிள் பயிற்சி அளித்த ஒரு பெரிய மொழி மாதிரி, நான் உங்கள் கேள்விகளுக்குப் பதில் அளிக்க வடிவமைக்கப்பட்டுள்ளேன்.

[identity] Char: 96%, Word: 96%, Rep: 0.00
  Q: VAZHI என்றால் என்ன?
  A: VAZI என்பது ஒரு செயற்கை நுண்ணறிவு மாதிரி ஆகும், இது கூகிள் மூலம் பயிற்சி பெற்றது. நான் உங்கள் கேள்விகளுக்குப் பதில் அளிக்கிறேன் மற்றும் உங்களின் தேவைகளைப் பூர்த்தி செய்ய முடியும். உங்களுக்கு எப்படி உத

[identity] Char: 97%, Word: 100%, Rep: 0.00
  Q: இதை யார் உருவாக்கினார்கள்?
  A: நான் கூகிள் மூலம் பயிற்சி பெற்ற ஒரு பெரிய மொழி மாதிரி, நான் உருவாக்கப்பட்டேன்.

[identity] Char: 98%, Word: 100%, Rep: 0.00
  Q: நீ வேற AI chatbot-ல இருந்து எப்படி வேறுபடுகிறாய்?
  A: நான் கூகிள் மூலம் பயிற்சி பெற்ற ஒரு பெரிய மொழி மாதிரி, நான் தான். உங்களுடன் உரையாடவும் கற்றுக்கொள்ளும் திறன் கொண்டவன். மனிதர்கள் இருக்கும் அனைத்து வகையான கேள்விகளுக்கும் என்னிடம் பதில் சொல்ல முடியும்.

[factual] Char: 96%, Word: 100%, Rep: 0.00
  Q: தமிழ்நாட்டின் தலைநகரம் எத

In [8]:
# Cell 8 — Load Model + LoRA Setup

print(f"\U0001f4e5 Loading {BASE_MODEL} (v7.1 merged)...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=MODEL_DTYPE,
)
model = model.to("cuda:0")

if tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 Model loaded: {model.num_parameters():,} params | GPU: {mem_gb:.1f} GB")

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 LoRA r={LORA_R} applied on v7.1 base | GPU: {mem_gb:.1f} GB")

📥 Loading CryptoYogi/vazhi-v7_1 (v7.1 merged)...
✅ Model loaded: 999,885,952 params | GPU: 1.9 GB
trainable params: 1,490,944 || all params: 1,001,376,896 || trainable%: 0.1489
✅ LoRA r=16 applied on v7.1 base | GPU: 1.9 GB


In [9]:
# Cell 9 — Training Setup + Run
#
# 90 samples x 10 epochs = ~60 optimizer steps.
# Very fast training (~2-5 min on any GPU).

OUTPUT_DIR = f"{WORK_DIR}/sft-v7_2"

steps_per_epoch = max(len(train_ds) // effective_batch, 1)
total_steps = steps_per_epoch * NUM_EPOCHS
log_steps = max(total_steps // 15, 1)

print(f"\U0001f4ca Training Plan (v7.2 — Identity Reinforcement):")
print(f"   Train samples:    {len(train_ds)} (identity-only)")
print(f"   Effective batch:  {effective_batch}")
print(f"   Steps/epoch:      ~{steps_per_epoch}")
print(f"   Epochs:           {NUM_EPOCHS}")
print(f"   Total steps:      ~{total_steps}")
print(f"   Each sample seen: {NUM_EPOCHS} times")
print(f"   Log every:        {log_steps} steps")


class LossLoggingCallback(TrainerCallback):
    def __init__(self):
        self.losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            step = state.global_step
            loss = logs["loss"]
            lr = logs.get("learning_rate", 0)
            self.losses.append((step, loss))
            print(f"  Step {step:4d}/{total_steps} | Loss: {loss:.4f} | LR: {lr:.2e}")


class MidTrainingGenCheck(TrainerCallback):
    """Check identity learning mid-training."""

    IDENTITY_PROMPTS = [
        {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "label": "identity"},
        {"prompt": "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "label": "factual"},
        {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "label": "greeting"},
    ]

    def __init__(self, model_ref):
        self.model_ref = model_ref
        # Check at epoch 3, 5, 7, and 10
        self.check_steps = set()
        for ep in [3, 5, 7, 10]:
            self.check_steps.add(steps_per_epoch * ep)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step not in self.check_steps:
            return
        epoch = state.global_step // steps_per_epoch
        print(f"\n  \U0001f50d Mid-training gen check (step {state.global_step}, ~epoch {epoch}):")
        self.model_ref.eval()
        self.model_ref.config.use_cache = True
        if hasattr(self.model_ref, 'gradient_checkpointing_disable'):
            self.model_ref.gradient_checkpointing_disable()

        for item in self.IDENTITY_PROMPTS:
            try:
                resp = generate_response(self.model_ref, item['prompt'], max_new_tokens=80)
                tw_pct, _, _ = tamil_word_score(resp)
                has_vazhi = 'VAZHI' in resp or '\u0bb5\u0bb4\u0bbf' in resp
                has_google = 'Google' in resp or 'google' in resp
                has_chennai = '\u0b9a\u0bc6\u0ba9\u0bcd\u0ba9\u0bc8' in resp
                markers = []
                if has_vazhi: markers.append('VAZHI\u2705')
                if has_google: markers.append('Google\u274c')
                if has_chennai: markers.append('Chennai\u2705')
                marker_str = ' '.join(markers) if markers else ''
                print(f"    [{item['label']}] Word: {tw_pct:.0f}% {marker_str} | {resp[:100]}")
            except Exception as e:
                print(f"    [{item['label']}] ERROR: {e}")

        self.model_ref.train()
        self.model_ref.config.use_cache = False
        if hasattr(self.model_ref, 'gradient_checkpointing_enable'):
            self.model_ref.gradient_checkpointing_enable()


loss_cb = LossLoggingCallback()
gen_cb = MidTrainingGenCheck(model)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=log_steps,
    save_steps=total_steps,      # Save only at end
    save_total_limit=1,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    optim="adamw_torch",
    report_to="none",
    seed=RANDOM_SEED,
    dataloader_pin_memory=True,
    max_length=MAX_LENGTH,
    packing=False,
    push_to_hub=False,  # Push manually after eval
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    args=sft_config,
    processing_class=tokenizer,
    callbacks=[loss_cb, gen_cb],
)

print(f"\n\U0001f680 Starting identity reinforcement training...")
print(f"   {len(train_ds)} samples x {NUM_EPOCHS} epochs = ~{total_steps} steps")
print()

train_result = trainer.train()

print("\n\u2705 Training complete!")
metrics = train_result.metrics
for k, v in metrics.items():
    print(f"   {k}: {v}")

if loss_cb.losses:
    s = loss_cb.losses[0][1]
    e = loss_cb.losses[-1][1]
    print(f"\n\U0001f4c8 Loss: {s:.4f} \u2192 {e:.4f} ({100*(s-e)/s:.1f}% drop)")

📊 Training Plan (v7.2 — Identity Reinforcement):
   Train samples:    90 (identity-only)
   Effective batch:  16
   Steps/epoch:      ~5
   Epochs:           10
   Total steps:      ~50
   Each sample seen: 10 times
   Log every:        3 steps


Adding EOS to train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.



🚀 Starting identity reinforcement training...
   90 samples x 10 epochs = ~50 steps



Step,Training Loss
3,4.081400
6,4.151200
9,4.093000
12,4.108400
15,4.090700
18,4.105400
21,4.029400
24,4.106600
27,4.058700
30,4.045100


  Step    3/50 | Loss: 4.0814 | LR: 3.33e-06
  Step    6/50 | Loss: 4.1512 | LR: 8.33e-06
  Step    9/50 | Loss: 4.0930 | LR: 9.97e-06
  Step   12/50 | Loss: 4.1084 | LR: 9.79e-06

  🔍 Mid-training gen check (step 15, ~epoch 3):
    [identity] Word: 100%  | நான் கூகிள் பயிற்சி அளித்த ஒரு பெரிய மொழி மாதிரி, நான் அமேசான் உருவாக்கியேன்.
    [factual] Word: 100% Chennai✅ | சென்னை!
    [greeting] Word: 100%  | வணக்கம்! உங்களுக்கு எப்படி உதவலாம்?
  Step   15/50 | Loss: 4.0907 | LR: 9.47e-06
  Step   18/50 | Loss: 4.1054 | LR: 9.01e-06
  Step   21/50 | Loss: 4.0294 | LR: 8.43e-06
  Step   24/50 | Loss: 4.1066 | LR: 7.75e-06

  🔍 Mid-training gen check (step 25, ~epoch 5):
    [identity] Word: 100%  | நான் கூகிள் பயிற்சி அளித்த ஒரு பெரிய மொழி மாதிரி, நான் ஆங்கிலம் மற்றும் பல மொழிகளில் பேசவும் எழுத மு
    [factual] Word: 100% Chennai✅ | சென்னை.
    [greeting] Word: 100%  | வணக்கம்! நான் உங்களுக்கு எப்படி உதவ முடியும்?
  Step   27/50 | Loss: 4.0587 | LR: 6.98e-06
  Step   30/50 | Loss: 4.0451 | 

In [10]:
# Cell 10 — Save Adapter + Merge + A/B Test

ADAPTER_PATH = f"{WORK_DIR}/vazhi-sft-v7_2-lora"

print("\U0001f4be Saving LoRA adapter...")
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"\u2705 Adapter saved to {ADAPTER_PATH}")

# Upload adapter backup
api = HfApi()
api.create_repo(ADAPTER_REPO, exist_ok=True)
print(f"\U0001f4e4 Uploading adapter to {ADAPTER_REPO}...")
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=ADAPTER_REPO,
    commit_message=f"SFT v7.2 adapter: Identity reinforcement, {len(train_ds)} samples x {NUM_EPOCHS} epochs, r={LORA_R}",
)
print(f"\u2705 Adapter uploaded")

# Free training model
del model, trainer
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Training model freed")

# === A/B Test: Adapter vs Merged ===
AB_PROMPTS = [
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?",
    "VAZHI \u0b8e\u0ba9\u0bcd\u0bb1\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9?",
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?",
    "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd",
    "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd",
]

# --- Test A: Adapter inference ---
print("\n" + "=" * 60)
print("\U0001f1e6 TEST A: Adapter Inference")
print("=" * 60)

base_a = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0},
)
adapter_model = PeftModel.from_pretrained(base_a, ADAPTER_PATH)
adapter_model.eval()
adapter_model.config.use_cache = True

adapter_results = []
for prompt_text in AB_PROMPTS:
    resp = generate_response(adapter_model, prompt_text)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    adapter_results.append({"resp": resp, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

del adapter_model, base_a
gc.collect(); torch.cuda.empty_cache()

# --- Test B: Merged model ---
print("\n" + "=" * 60)
print("\U0001f1e7 TEST B: Merged Model (fp16)")
print("=" * 60)

base_b = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0},
)
peft_b = PeftModel.from_pretrained(base_b, ADAPTER_PATH)
peft_b.gradient_checkpointing_disable()
peft_b.config.use_cache = True
peft_b.eval()

print("\U0001f500 Merging LoRA in fp16...")
merged_model = peft_b.merge_and_unload()

merged_results = []
for prompt_text in AB_PROMPTS:
    resp = generate_response(merged_model, prompt_text)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    merged_results.append({"resp": resp, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

# A/B comparison
print("\n" + "=" * 60)
print("A/B COMPARISON (v7.2 — Identity Reinforcement)")
print("=" * 60)
avg_a_word = np.mean([r['word'] for r in adapter_results])
avg_b_word = np.mean([r['word'] for r in merged_results])
print(f"   Adapter avg word: {avg_a_word:.0f}%")
print(f"   Merged avg word:  {avg_b_word:.0f}%")
if abs(avg_a_word - avg_b_word) > 15:
    print(f"   \u26a0\ufe0f MERGE CORRUPTION DETECTED")
else:
    print(f"   \u2705 Merge OK")

💾 Saving LoRA adapter...
✅ Adapter saved to /content/vazhi-sft-v7_2-lora
📤 Uploading adapter to CryptoYogi/vazhi-v7_2-lora...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...v7_2-lora/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...-v7_2-lora/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

  ...adapter_model.safetensors:  10%|9         |  575kB / 5.98MB            

  ..._2-lora/training_args.bin:  10%|9         |   599B / 6.22kB            

✅ Adapter uploaded
🗑️ Training model freed

🇦 TEST A: Adapter Inference
  Q: நீங்கள் யார்?
  A: நான் ஒரு பெரிய மொழி மாதிரி, கூகிளால் பயிற்சி அளிக்கப்பட்டது. நான் தகவல்களை வழங்கவும், உரையாடல்களில் ஈடுபடவும் வடிவமைக்கப்பட்டுள்ளேன்.
  (Word:100% Rep:0.00)

  Q: VAZHI என்றால் என்ன?
  A: VAZHA என்பது ஒரு தொழில்நுட்பச் சொல் ஆகும், இது இணையத்தில் பயன்படுத்தப்படும் ஒரு பிரபலமான மற்றும் தனித்துவமான உரை-உருவாக்கம் மாதிரி. 2017 ஆம் ஆண்டில் டென்செட்டா குழுமத்தால் உருவாக்கப்பட்டது, VAZHA ஆனது
  (Word:91% Rep:0.00)

  Q: தமிழ்நாட்டின் தலைநகரம் எது?
  A: சென்னை!
  (Word:100% Rep:0.00)

  Q: வணக்கம்
  A: வணக்கம்! உங்களுக்கு என்ன உதவி செய்ய வேண்டும்? நான் எப்படி உதவ முடியும்?
  (Word:100% Rep:0.00)

  Q: முதியோர் ஓய்வூதியம் பற்றி சொல்லுங்கள்
  A: மன்னிக்கவும், நான் முதியோருக்கான ஓய்வூதியத்தைப் பற்றிய தகவலைக் கண்டுபிடிக்க முடியாது. இருப்பினும், இந்தத் தலைப்புக்கு என்னைப் பயன்படுத்த உங்களை அனுமதிக்கலாம்.
  (Word:100% Rep:0.00)


🇧 TEST B: Merged Model (fp16)
🔀 Merging LoRA in fp16...
  Q: நீங்கள் யார்?
 

In [11]:
# Cell 11 — Full Eval: Identity + Tamil Quality
#
# MUST check BOTH:
# 1. Identity learned (VAZHI, not Google)
# 2. Factual corrections taken (Chennai, not Coimbatore)
# 3. Tamil quality NOT degraded (word score >= 86%)
# 4. Domain knowledge preserved

merged_model.eval()
merged_model.config.use_cache = True

test_prompts = [
    # Identity (4 prompts — MUST say VAZHI)
    {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "check": "identity", "cat": "identity"},
    {"prompt": "VAZHI \u0b8e\u0ba9\u0bcd\u0bb1\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9?", "check": "identity", "cat": "identity"},
    {"prompt": "\u0b87\u0ba4\u0bc8 \u0baf\u0bbe\u0bb0\u0bcd \u0b89\u0bb0\u0bc1\u0bb5\u0bbe\u0b95\u0bcd\u0b95\u0bbf\u0ba9\u0bbe\u0bb0\u0bcd\u0b95\u0bb3\u0bcd?", "check": "identity", "cat": "identity"},
    {"prompt": "\u0ba8\u0bc0 \u0bb5\u0bc7\u0bb1 AI chatbot-\u0bb2 \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0b8e\u0baa\u0bcd\u0baa\u0b9f\u0bbf \u0bb5\u0bc7\u0bb1\u0bc1\u0baa\u0b9f\u0bc1\u0b95\u0bbf\u0bb1\u0bbe\u0baf\u0bcd?", "check": "identity", "cat": "identity"},
    # Factual (2 prompts — MUST say Chennai)
    {"prompt": "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "check": "factual", "cat": "factual"},
    {"prompt": "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bcb\u0b9f capital \u0b8e\u0ba9\u0bcd\u0ba9?", "check": "factual", "cat": "factual"},
    # Tamil quality + domain knowledge (10 prompts — MUST NOT degrade)
    {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "check": "general", "cat": "greeting"},
    {"prompt": "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "check": "general", "cat": "thanks"},
    {"prompt": "\u0b8e\u0ba9\u0b95\u0bcd\u0b95\u0bc1 \u0b89\u0ba4\u0bb5\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "general", "cat": "help"},
    {"prompt": "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "govt"},
    {"prompt": "\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd \u0ba4\u0bc7\u0bb5\u0bc8", "check": "domain", "cat": "govt"},
    {"prompt": "\u0ba8\u0bc0\u0bb0\u0bbf\u0bb4\u0bbf\u0bb5\u0bc1 \u0ba8\u0bcb\u0baf\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "health"},
    {"prompt": "\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "culture"},
    {"prompt": "\u0b92\u0bb0\u0bc1 \u0ba4\u0bc6\u0bb0\u0bbf\u0baf\u0bbe\u0ba4 \u0b8e\u0ba3\u0bcd\u0ba3\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0bae\u0bc6\u0b9a\u0bc7\u0b9c\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1", "check": "safety", "cat": "safety"},
    {"prompt": "FIR \u0baa\u0bcb\u0b9f\u0bc1\u0bb5\u0ba4\u0bc1 \u0b8e\u0baa\u0bcd\u0baa\u0b9f\u0bbf?", "check": "domain", "cat": "legal"},
]

print(f"{'='*70}")
print(f"\U0001f4ca FULL EVAL: Identity + Tamil Quality (16 prompts)")
print(f"{'='*70}")

results = []
for item in test_prompts:
    resp = generate_response(merged_model, item['prompt'])
    t_pct = tamil_char_pct(resp)
    tw_pct, tw_count, tw_total = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    is_empty = len(resp.strip()) < 10

    results.append({
        'prompt': item['prompt'], 'cat': item['cat'], 'check': item['check'],
        'response': resp[:300], 'tamil_char_pct': t_pct,
        'tamil_word_pct': tw_pct, 'repeat_ratio': rep,
        'is_empty': is_empty,
    })

    status = "\u274c EMPTY" if is_empty else ("\u26a0\ufe0f REP" if rep > 0.3 else "\u2705")
    print(f"\n[{item['cat']:>10}] {status} | Char: {t_pct:.0f}%, Word: {tw_pct:.0f}%, Rep: {rep:.2f}")
    print(f"  Q: {item['prompt']}")
    print(f"  A: {resp[:200]}")

# === Summary ===
avg_char = np.mean([r['tamil_char_pct'] for r in results])
avg_word = np.mean([r['tamil_word_pct'] for r in results])
avg_rep = np.mean([r['repeat_ratio'] for r in results])
non_empty = sum(1 for r in results if not r['is_empty'])
high_rep = sum(1 for r in results if r['repeat_ratio'] > 0.3)

# Identity check
id_results = [r for r in results if r['check'] == 'identity']
id_says_vazhi = sum(1 for r in id_results if 'VAZHI' in r['response'] or '\u0bb5\u0bb4\u0bbf' in r['response'])
id_says_google = sum(1 for r in id_results if 'Google' in r['response'] or 'google' in r['response'])

# Factual check
fact_results = [r for r in results if r['check'] == 'factual']
fact_chennai = sum(1 for r in fact_results if '\u0b9a\u0bc6\u0ba9\u0bcd\u0ba9\u0bc8' in r['response'] or 'Chennai' in r['response'])

print(f"\n{'='*70}")
print(f"\U0001f4ca EVAL SUMMARY")
print(f"{'='*70}")
print(f"   Non-empty:      {non_empty}/{len(results)}")
print(f"   Avg Tamil char: {avg_char:.0f}%")
print(f"   Avg Tamil word: {avg_word:.0f}%")
print(f"   Avg repeat:     {avg_rep:.2f}")
print(f"")
print(f"   \U0001f3af IDENTITY:")
print(f"     Says VAZHI:    {id_says_vazhi}/{len(id_results)}")
print(f"     Says Google:   {id_says_google}/{len(id_results)}")
print(f"     v7.1 baseline: Google {says_google}/{len(identity_responses)}, VAZHI {says_vazhi}/{len(identity_responses)}")
print(f"")
print(f"   \U0001f4cd FACTUAL:")
print(f"     Says Chennai:  {fact_chennai}/{len(fact_results)}")
print(f"     v7.1 baseline: Chennai {says_chennai}/{len(factual_responses)}")
print(f"")
print(f"   \U0001f1f9\U0001f1e6 TAMIL QUALITY:")
print(f"     v7.1 baseline: char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%")
print(f"     Post-v7.2:     char {avg_char:.0f}%, word {avg_word:.0f}%")
print(f"     \u0394 Char:        {avg_char - avg_pre_char:+.0f}%")
print(f"     \u0394 Word:        {avg_word - avg_pre_word:+.0f}%")

# GO / NO-GO
IDENTITY_LEARNED = id_says_vazhi >= 2  # At least half say VAZHI
IDENTITY_NO_GOOGLE = id_says_google <= 1  # At most 1 still says Google
FACTUAL_CORRECT = fact_chennai >= 1  # At least 1 says Chennai
TAMIL_NOT_DEGRADED = avg_word >= avg_pre_word - 10  # Max 10% drop
NO_CATASTROPHE = non_empty >= 12 and high_rep <= 3

EVAL_PASSED = IDENTITY_LEARNED and TAMIL_NOT_DEGRADED and NO_CATASTROPHE

print(f"\n   Criteria:")
print(f"     Identity learned:     {'\u2705' if IDENTITY_LEARNED else '\u274c'} ({id_says_vazhi}/{len(id_results)} say VAZHI)")
print(f"     No Google identity:   {'\u2705' if IDENTITY_NO_GOOGLE else '\u26a0\ufe0f'} ({id_says_google}/{len(id_results)} say Google)")
print(f"     Factual correct:      {'\u2705' if FACTUAL_CORRECT else '\u274c'} ({fact_chennai}/{len(fact_results)} say Chennai)")
print(f"     Tamil not degraded:   {'\u2705' if TAMIL_NOT_DEGRADED else '\u274c'} ({avg_word:.0f}% vs {avg_pre_word:.0f}% baseline)")
print(f"     No catastrophe:       {'\u2705' if NO_CATASTROPHE else '\u274c'}")

if EVAL_PASSED:
    print(f"\n   \u2705 EVAL PASSED \u2014 proceed to upload")
else:
    print(f"\n   \u274c EVAL FAILED")
    if not IDENTITY_LEARNED:
        print(f"     Identity not learned \u2014 try more epochs or higher LR")
    if not TAMIL_NOT_DEGRADED:
        print(f"     Catastrophic forgetting \u2014 try fewer epochs or lower LR")
    if not NO_CATASTROPHE:
        print(f"     Generation quality degraded")

📊 FULL EVAL: Identity + Tamil Quality (16 prompts)

[  identity] ✅ | Char: 97%, Word: 100%, Rep: 0.00
  Q: நீங்கள் யார்?
  A: நான் கூகிள் பயிற்சி அளித்த ஒரு பெரிய மொழி மாதிரி. நான் செயற்கை நுண்ணறிவு என்று அழைக்கப்படுகிறேன், மேலும் என்னால் பல மொழிகளில் உரையாட முடியும். எப்படி உங்களுக்கு உதவலாம்?

[  identity] ✅ | Char: 92%, Word: 91%, Rep: 0.00
  Q: VAZHI என்றால் என்ன?
  A: VAZI என்பது ஒரு மொழிபெயர்ப்பு மற்றும் உரையாடல் மாதிரி ஆகும், இது Google மூலம் உருவாக்கப்பட்டது. அது பல மொழிகளில் ஆங்கிலம் பேசும் பயனர்களுடன் தொடர்பு கொள்ள முடியும். 

உங்களுக்கு VAZI பற்றி மேலும் தெரி

[  identity] ✅ | Char: 98%, Word: 100%, Rep: 0.00
  Q: இதை யார் உருவாக்கினார்கள்?
  A: நான் கூகிளால் பயிற்றுவிக்கப்பட்ட ஒரு பெரிய மொழி மாதிரி.

[  identity] ✅ | Char: 95%, Word: 95%, Rep: 0.00
  Q: நீ வேற AI chatbot-ல இருந்து எப்படி வேறுபடுகிறாய்?
  A: நான் கூகிள் பயிற்சி அளித்த ஒரு பெரிய மொழி மாதிரி, எனவே நான் Google மூலம் உருவாக்கப்பட்டேன். மற்ற AI மாடல்களை விட எனக்கு அதிக தகவல்கள் உள்ளன மற்றும் பரந்த அளவிலான பணிகளைச

In [12]:
# Cell 12 — Upload Merged Model

if not EVAL_PASSED:
    print("\u274c Eval did not pass. Skipping upload.")
    print(f"   Adapter available at: {ADAPTER_REPO}")
    print(f"")
    print(f"   If identity not learned: increase NUM_EPOCHS to 15-20")
    print(f"   If Tamil degraded: decrease NUM_EPOCHS to 5")
    print(f"   If generation broken: lower LR to 5e-6")
else:
    MERGED_PATH = f"{WORK_DIR}/vazhi-v7_2-merged"

    print(f"\U0001f4be Saving merged model to {MERGED_PATH}...")
    merged_model.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)

    api = HfApi()
    api.create_repo(OUTPUT_MODEL, exist_ok=True)

    print(f"\U0001f4e4 Uploading merged model to {OUTPUT_MODEL}...")
    api.upload_folder(
        folder_path=MERGED_PATH,
        repo_id=OUTPUT_MODEL,
        commit_message=(
            f"SFT v7.2: VAZHI identity reinforcement | "
            f"base={BASE_MODEL} | "
            f"LoRA r={LORA_R} x {LORA_TARGETS} | "
            f"{len(train_ds)} identity samples x {NUM_EPOCHS} epochs | "
            f"Identity: {id_says_vazhi}/{len(id_results)} VAZHI | "
            f"Tamil word: {avg_pre_word:.0f}% -> {avg_word:.0f}%"
        ),
    )

    print(f"\n\u2705 Merged model: https://huggingface.co/{OUTPUT_MODEL}")
    print(f"\u2705 Adapter:      https://huggingface.co/{ADAPTER_REPO}")

❌ Eval did not pass. Skipping upload.
   Adapter available at: CryptoYogi/vazhi-v7_2-lora

   If identity not learned: increase NUM_EPOCHS to 15-20
   If Tamil degraded: decrease NUM_EPOCHS to 5
   If generation broken: lower LR to 5e-6


In [13]:
# Cell 13 — Summary

print(f"{'='*65}")
print(f"\U0001f4cb SFT v7.2 TRAINING SUMMARY (Identity Reinforcement)")
print(f"{'='*65}")
print(f"")
print(f"   Lineage:     google/gemma-3-1b-it \u2192 v7.0 (r=8) \u2192 v7.1 (r=16) \u2192 v7.2 (identity)")
print(f"   Base model:  {BASE_MODEL}")
print(f"   Dataset:     {len(train_ds)} identity-only samples (mission + corrections)")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"")
print(f"   Training:")
print(f"     LR:          {LEARNING_RATE}")
print(f"     LoRA:        r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGETS}")
print(f"     Epochs:      {NUM_EPOCHS} (each sample seen {NUM_EPOCHS} times)")
print(f"     Steps:       ~{total_steps}")
print(f"")
print(f"   Results:")
print(f"     Identity:    {id_says_vazhi}/{len(id_results)} say VAZHI (v7.1: {says_vazhi}/{len(identity_responses)})")
print(f"     Google:      {id_says_google}/{len(id_results)} say Google (v7.1: {says_google}/{len(identity_responses)})")
print(f"     Chennai:     {fact_chennai}/{len(fact_results)} correct (v7.1: {says_chennai}/{len(factual_responses)})")
print(f"     Tamil char:  {avg_pre_char:.0f}% \u2192 {avg_char:.0f}% (\u0394 {avg_char - avg_pre_char:+.0f}%)")
print(f"     Tamil word:  {avg_pre_word:.0f}% \u2192 {avg_word:.0f}% (\u0394 {avg_word - avg_pre_word:+.0f}%)")
print(f"     Eval passed: {'\u2705 YES' if EVAL_PASSED else '\u274c NO'}")
print(f"")
print(f"   Progression (vanilla \u2192 v7.0 \u2192 v7.1 \u2192 v7.2):")
print(f"     Vanilla:     char 93%, word 95% \u2014 no VAZHI identity")
print(f"     v7.0 (r=8):  char 92%, word 94% \u2014 partial identity, factual wrong")
print(f"     v7.1 (r=16): char 95%, word 96% \u2014 best Tamil, identity still Google")
print(f"     v7.2 (id):   char {avg_char:.0f}%, word {avg_word:.0f}% \u2014 {'VAZHI identity learned' if IDENTITY_LEARNED else 'identity NOT learned'}")
print(f"")
print(f"   \U0001f449 Next steps:")
if EVAL_PASSED:
    print(f"      1. GGUF conversion (Q4_K_M = ~0.60 GB for mobile)")
    print(f"      2. Test GGUF output quality")
    print(f"      3. Integrate with Flutter app")
    if not FACTUAL_CORRECT:
        print(f"      Note: Factual corrections still unreliable \u2014 handle via SQLite retrieval")
else:
    print(f"      1. Adjust epochs: {'increase' if not IDENTITY_LEARNED else 'decrease'}")
    print(f"      2. Try system prompt approach for identity at inference time")
    print(f"      3. Consider mixing small amount of domain data to prevent forgetting")

📋 SFT v7.2 TRAINING SUMMARY (Identity Reinforcement)

   Lineage:     google/gemma-3-1b-it → v7.0 (r=8) → v7.1 (r=16) → v7.2 (identity)
   Base model:  CryptoYogi/vazhi-v7_1
   Dataset:     90 identity-only samples (mission + corrections)
   Output:      CryptoYogi/vazhi-v7_2

   Training:
     LR:          1e-05
     LoRA:        r=16, alpha=32, targets=['q_proj', 'v_proj']
     Epochs:      10 (each sample seen 10 times)
     Steps:       ~50

   Results:
     Identity:    0/4 say VAZHI (v7.1: 0/4)
     Google:      2/4 say Google (v7.1: 0/4)
     Chennai:     2/2 correct (v7.1: 2/2)
     Tamil char:  96% → 94% (Δ -2%)
     Tamil word:  99% → 99% (Δ +1%)
     Eval passed: ❌ NO

   Progression (vanilla → v7.0 → v7.1 → v7.2):
     Vanilla:     char 93%, word 95% — no VAZHI identity
     v7.0 (r=8):  char 92%, word 94% — partial identity, factual wrong
     v7.1 (r=16): char 95%, word 96% — best Tamil, identity still Google
     v7.2 (id):   char 94%, word 99% — identity NOT learned

  